# Nemotron v7.3 — Plain LoRA + v9 Data (Fast)

**Reverts the broken DoRA/rsLoRA/PiSSA stack.** Trains a stock LoRA
adapter that vLLM can load without surprises, on the new
`train_cot_v9_real_transform.jsonl` dataset (transformation puzzles
now ship with real symbolic-derivation CoT).

### Speed optimizations (vs v7/v7.2)
| Change | Impact |
|---|---|
| `MAX_SEQ_LEN` 7680 → 4096 | ~2× faster (attention is O(n²)) |
| `packing=True` | ~2-3× faster (no padding waste) |
| `BATCH_SIZE` 4 → 8 | better GPU utilization |
| `GRAD_ACCUM` 8 → 2 | same effective batch (16) but fewer accum steps |
| `gradient_checkpointing=False` | ~30% faster (96 GB VRAM has room) |
| Plain LoRA (no DoRA/rsLoRA/PiSSA) | no inference-time mismatch |

### Eval-server contract
| | |
|---|---|
| `max_lora_rank` | 32 |
| `max_tokens` | 7680 |
| `temperature` | 0.0 |
| `lora_dropout` | must be 0 |


In [ ]:
# ============================================================
# 1. OFFLINE DEPENDENCY INSTALLATION
# ============================================================
import subprocess, sys, os
from pathlib import Path

def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read().strip()
            rel_pack_path = pth_file.parent / relpath
            if rel_pack_path.exists():
                sys.path.append(str(rel_pack_path))

offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir  = "/kaggle/working/packages"
os.makedirs(target_dir, exist_ok=True)

resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl", "peft"
    ])
    print("Offline packages installed.")

os.environ["WANDB_MODE"] = "offline"
WANDB_AVAILABLE = False
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir, "wandb"
    ])
    WANDB_AVAILABLE = True
    print("wandb installed (offline).")
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb"])
        WANDB_AVAILABLE = True
        print("wandb installed (online).")
    except Exception:
        print("wandb not available")

sys.path.append(target_dir)
resolve_python_path(target_dir)


In [ ]:
# ============================================================
# 2. IMPORTS
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import stat, shutil, zipfile, time, json, re
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
from collections import Counter

if WANDB_AVAILABLE:
    import wandb

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"W&B     : {'offline mode' if WANDB_AVAILABLE else 'disabled'}")


In [ ]:
# ============================================================
# 2b. W&B OFFLINE INIT
# ============================================================
WANDB_PROJECT  = "nemotron-v73"
WANDB_RUN_NAME = f"v73-plain-lora-r32-{time.strftime('%Y%m%d-%H%M%S')}"
WANDB_DIR      = "/kaggle/working"

if WANDB_AVAILABLE:
    wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, dir=WANDB_DIR,
               mode="offline", tags=["nemotron", "lora", "v73", "plain"])
    print(f"W&B run: {WANDB_RUN_NAME}")


In [ ]:
# ============================================================
# 3. TRITON FIXES (rmsnorm + ptxas-blackwell binary copy)
# ============================================================
# Two fixes:
#  (a) rmsnorm_fn replaced with pure PyTorch (mamba_ssm Triton kernel
#      fails on some GPU configs)
#  (b) ptxas-blackwell binary lives in a read-only mount without +x;
#      copy it to /tmp and chmod, then redirect Triton's env vars.

# (a) rmsnorm_fn patch
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    var = x.pow(2).mean(-1, keepdim=True)
    y = x * torch.rsqrt(var + eps)
    out = y * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# (b) ptxas-blackwell binary fix — search BOTH path conventions
import glob
candidates = (
    glob.glob("/kaggle/usr/lib/notebooks/**/ptxas-blackwell", recursive=True)
    + glob.glob("/kaggle/usr/lib/notebooks/**/ptxas", recursive=True)
    + glob.glob("/usr/local/cuda*/bin/ptxas", recursive=True)
)
src = next((c for c in candidates if "blackwell" in c), None) \
   or (candidates[0] if candidates else None)

if src and os.path.exists(src):
    dst = "/tmp/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
              "TRITON_PTXAS_BIN", "TRITON_PTXAS"):
        os.environ[v] = dst

    # Bust Triton's caches
    try:
        import triton.backends.nvidia.compiler as nv_compiler
        try: nv_compiler.get_ptxas_version.cache_clear()
        except AttributeError: pass
        nv_compiler.get_ptxas_version = lambda arch: "release 12.8"
        from triton import knobs as triton_knobs
        for attr in ("ptxas", "ptxas_blackwell"):
            triton_knobs.nvidia.__dict__.pop(attr, None)
    except Exception as e:
        print(f"Triton cache clear warning: {e}")
    print(f"[ok] ptxas -> {dst}")
else:
    print("[warn] no ptxas binary found — will likely crash on Mamba kernel")


In [ ]:
# ============================================================
# 4. HYPERPARAMETERS — v7.3 fast plain-LoRA
# ============================================================
LORA_RANK       = 32
LORA_ALPHA      = 64
MAX_SEQ_LEN     = 4096            # fits 99% of CoT samples
NUM_EPOCHS      = 4               # plain LoRA can take 4 epochs
BATCH_SIZE      = 8               # bigger batch for utilization
GRAD_ACCUM      = 2               # effective batch = 16
LR              = 1e-4            # standard plain-LoRA LR
WARMUP_STEPS    = 50
SAVE_EVERY_N_EPOCHS = 1

# Speed: skip gradient checkpointing (96 GB VRAM has room)
GRAD_CKPT       = False

MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR = "/kaggle/working/adapter"
CKPT_DIR   = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# v9 data with real symbolic CoT for transformation
DATA_PATHS = [
    "/kaggle/input/nemotron-cot-v9/train_cot_v9_real_transform.jsonl",
    "/kaggle/input/nemotron-dataset/train_cot_v9_real_transform.jsonl",
    "/kaggle/input/nemotron-cot-v8/train_cot_v8_merged.jsonl",
    "/kaggle/input/nemotron-cot-v5/train_cot_v5_merged.jsonl",
]

# Eval-server assertions
assert LORA_RANK   <= 32,   f"LORA_RANK={LORA_RANK} exceeds eval cap"
assert MAX_SEQ_LEN <= 7680, f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds eval cap"

print(f"Rank/alpha : {LORA_RANK}/{LORA_ALPHA}  | seqlen {MAX_SEQ_LEN}")
print(f"Epochs     : {NUM_EPOCHS}  | LR {LR:.1e}")
print(f"Batch      : {BATCH_SIZE}×{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} eff")
print(f"GradCkpt   : {GRAD_CKPT}  (skip = ~30% faster)")
print(f"Packing    : True (~2-3× faster)")


In [ ]:
# ============================================================
# 5. CALLBACKS — progress + per-epoch checkpoint zip
# ============================================================
class LiveProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="Training", dynamic_ncols=True)
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not self.pbar or not logs: return
        self.pbar.update(state.global_step - self.pbar.n)
        msg = []
        if "loss" in logs: msg.append(f"loss={logs['loss']:.4f}")
        if "learning_rate" in logs: msg.append(f"lr={logs['learning_rate']:.2e}")
        if msg: self.pbar.set_postfix_str(" ".join(msg))
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar: self.pbar.close()


class CheckpointZipCallback(TrainerCallback):
    def __init__(self, ckpt_dir, every_n=1):
        self.ckpt_dir = ckpt_dir
        self.every_n  = every_n
        self.epoch_losses = {}
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.epoch_losses[int(state.epoch)] = logs["loss"]
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch)
        if epoch == 0 or epoch % self.every_n != 0: return
        edir = os.path.join(self.ckpt_dir, f"adapter_epoch_{epoch:02d}")
        os.makedirs(edir, exist_ok=True)
        model.save_pretrained(edir)
        # Patch base model name
        cfg_path = os.path.join(edir, "adapter_config.json")
        if os.path.exists(cfg_path):
            with open(cfg_path) as f: cfg = json.load(f)
            cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
            with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)
        zip_path = os.path.join(self.ckpt_dir, f"adapter_epoch_{epoch:02d}.zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fn in os.listdir(edir):
                fp = os.path.join(edir, fn)
                if os.path.isfile(fp):
                    zf.write(fp, arcname=fn)
        sz = os.path.getsize(zip_path) / 1024 / 1024
        loss = self.epoch_losses.get(epoch, float('nan'))
        print(f"[ckpt] epoch {epoch:2d} | loss={loss:.4f} | {zip_path} ({sz:.1f} MB)")
    def print_summary(self):
        if not self.epoch_losses: return
        best = min(self.epoch_losses.items(), key=lambda x: x[1])
        print("\n=== Checkpoints (lowest loss = best candidate) ===")
        for e, l in sorted(self.epoch_losses.items()):
            tag = "  <-- BEST" if e == best[0] else ""
            print(f"  epoch {e}  loss={l:.4f}{tag}")

ckpt_callback = CheckpointZipCallback(CKPT_DIR, every_n=SAVE_EVERY_N_EPOCHS)


In [ ]:
# ============================================================
# 6. LOAD DATA (prefer v9 — has real symbolic CoT for transformations)
# ============================================================
data = []
data_path = None
for path in DATA_PATHS:
    if os.path.exists(path):
        data_path = path
        with open(path) as f:
            for line in f:
                data.append(json.loads(line))
        print(f"Loaded {len(data)} examples from {path}")
        break

if not data:
    raise FileNotFoundError("No training data found in any candidate path")

print(f"Source: {data_path}")


In [ ]:
# ============================================================
# 7. TOKENIZER + FORMAT
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

EVAL_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

def infer_cat(prompt):
    p = prompt.lower()
    if "encryption rules" in p or "decrypt" in p: return "cipher"
    if "gravity" in p or "gravitational" in p: return "gravity"
    if "binary" in p or "8-bit" in p: return "bit_manipulation"
    if "roman" in p or "numeral" in p: return "numeral"
    if "unit" in p and ("convert" in p or "meter" in p or "gram" in p):
        return "unit_conversion"
    if "transform" in p: return "transformation"
    return "other"

texts, labels = [], []
for ex in data:
    msgs = [m for m in ex['messages'] if m['role'] != 'system']
    user = msgs[0]['content'] if msgs and msgs[0]['role'] == 'user' else ''
    cat = infer_cat(user)

    asst = msgs[-1]
    if '<think>' not in asst['content']:
        m = re.search(r'(\\boxed\{.*?\})\s*$', asst['content'])
        if m:
            reasoning = asst['content'][:m.start()].strip()
            msgs[-1] = {'role': 'assistant',
                        'content': f"<think>\n{reasoning}\n</think>\n{m.group(1)}"}
    try:
        text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                              add_generation_prompt=False)
    except Exception:
        text = (f"<|im_start|>user\n{msgs[0]['content']}<|im_end|>\n"
                f"<|im_start|>assistant\n{msgs[-1]['content']}<|im_end|>")

    texts.append(text)
    labels.append(cat)

# 2x upsample transformation (the weakest category)
upsample_transform = []
for t, l in zip(texts, labels):
    if l == 'transformation':
        upsample_transform.append((t, l))
texts.extend(t for t, _ in upsample_transform)
labels.extend(l for _, l in upsample_transform)
print(f"Upsampled {len(upsample_transform)} transformation puzzles 2× → total {len(texts)}")

print("\nCategory distribution after upsampling:")
for n, c in Counter(labels).most_common():
    print(f"  {n:20s} {c:5d}")

hf_dataset = Dataset.from_dict({'text': texts})


In [ ]:
# ============================================================
# 8. DROP OVERSIZED SAMPLES
# ============================================================
before = len(hf_dataset)
def _len(ex):
    return {'tl': len(tokenizer(ex['text'], truncation=False,
                                 return_attention_mask=False)['input_ids'])}
hf_dataset = hf_dataset.map(_len, desc="Counting tokens")
hf_dataset = hf_dataset.filter(lambda x: x['tl'] <= MAX_SEQ_LEN, desc="Filtering")
hf_dataset = hf_dataset.remove_columns(['tl'])
print(f"Kept {len(hf_dataset)}/{before} (dropped {before - len(hf_dataset)} > {MAX_SEQ_LEN} tokens)")


In [ ]:
# ============================================================
# 9. LOAD MODEL (bf16, no quantization)
# ============================================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, low_cpu_mem_usage=True,
)
if GRAD_CKPT:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
print(f"Model loaded: {type(model).__name__}  | grad_ckpt={GRAD_CKPT}")


In [ ]:
# ============================================================
# 10. APPLY LoRA — PLAIN (no DoRA, no rsLoRA, no PiSSA)
# ============================================================
# Reverts to the v7-baseline config that vLLM honors exactly.
# No experimental flags — every adapter math operation matches what
# the eval server's vLLM does at inference.

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
    "in_proj", "out_proj",                      # Mamba-2
    "up_proj", "down_proj",                     # MLP / MoE
    "lm_head",                                  # Output head
]

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    # No use_dora, no use_rslora, no init_lora_weights, no modules_to_save
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# ============================================================
# 11. TRAINING — packed SFT (FAST)
# ============================================================
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch_fused",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    save_strategy="no",
    report_to="wandb" if WANDB_AVAILABLE else "none",
    run_name=WANDB_RUN_NAME if WANDB_AVAILABLE else None,
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=True,                          # SPEED: pack samples
    gradient_checkpointing=GRAD_CKPT,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[LiveProgressCallback(), ckpt_callback],
)

print(f"\nStarting training:")
print(f"  {len(hf_dataset)} samples | {NUM_EPOCHS} epochs | LR={LR:.1e}")
print(f"  batch={BATCH_SIZE}×{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM} eff")
print(f"  W&B: {'offline' if WANDB_AVAILABLE else 'disabled'}\n")

t0 = time.time()
trainer.train()
print(f"\nTraining done in {(time.time()-t0)/3600:.2f} hrs")
ckpt_callback.print_summary()


In [ ]:
# ============================================================
# 12. SAVE FINAL ADAPTER (no PiSSA conversion needed)
# ============================================================
trainer.model.save_pretrained(OUTPUT_DIR)

cfg_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

print(f"r={cfg.get('r')}  alpha={cfg.get('lora_alpha')}")
print(f"target_modules={cfg.get('target_modules')}")
print(f"peft_type={cfg.get('peft_type')}")
print(f"use_dora={cfg.get('use_dora', False)} (should be False)")
print(f"use_rslora={cfg.get('use_rslora', False)} (should be False)")

for fn in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, fn)
    if os.path.isfile(fp):
        print(f"  {fn}  ({os.path.getsize(fp)/1e6:.2f} MB)")


In [ ]:
# ============================================================
# 13. ZIP FINAL ADAPTER
# ============================================================
ZIP_PATH = "/kaggle/working/adapter.zip"
if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(OUTPUT_DIR):
        fp = os.path.join(OUTPUT_DIR, fn)
        if os.path.isfile(fp):
            zf.write(fp, arcname=fn)

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"Final adapter: {ZIP_PATH} ({zip_mb:.1f} MB)")

print("\nPer-epoch checkpoints (pick lowest-loss to submit):")
for fn in sorted(os.listdir(CKPT_DIR)):
    if fn.endswith(".zip"):
        sz = os.path.getsize(os.path.join(CKPT_DIR, fn)) / 1024 / 1024
        print(f"  {fn}  ({sz:.1f} MB)")

if WANDB_AVAILABLE:
    wandb.finish()
    wandb_dir = os.path.join(WANDB_DIR, "wandb")
    if os.path.exists(wandb_dir):
        wz = "/kaggle/working/wandb_logs.zip"
        with zipfile.ZipFile(wz, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, _, files in os.walk(wandb_dir):
                for f_ in files:
                    fp = os.path.join(root, f_)
                    zf.write(fp, arcname=os.path.relpath(fp, WANDB_DIR))
        print(f"W&B logs: {wz}")
